In [ ]:
import yt 
import numpy as np

def _gamma(field, data):
    u1 = data[('gas', 'four_velocity_r')]
    u2 = data[('gas', 'four_velocity_theta')]
    u3 = data[('gas', 'four_velocity_z')]
    c = data.ds.quan(1.0, 'code_velocity')
    return (1 + (u1/c)**2 + (u2/c)**2 + (u3/c)**2)**0.5


yt.add_field(('gas', 'lorentz_gamma'), function=_gamma, units="", sampling_type = "cell", dimensions = yt.units.dimensionless)
# Cartesian velocity components derived directly from the native cylindrical fields.
# ('athena_pp', 'vel1') = v_R,  ('athena_pp', 'vel2') = v_phi
# ('index', 'theta')    = azimuthal angle phi of each cell
# v_x = v_R*cos(phi) - v_phi*sin(phi)
# v_y = v_R*sin(phi) + v_phi*cos(phi)
def _vel_cyl_x(field, data):
    phi  = data[('index', 'theta')]
    vr   = data[('athena_pp', 'vel1')]
    vphi = data[('athena_pp', 'vel2')]
    return vr * np.cos(phi) - vphi * np.sin(phi)

def _vel_cyl_y(field, data):
    phi  = data[('index', 'theta')]
    vr   = data[('athena_pp', 'vel1')]
    vphi = data[('athena_pp', 'vel2')]
    return vr * np.sin(phi) + vphi * np.cos(phi)

yt.add_field(('gas', 'vel_cyl_x'), function=_vel_cyl_x,
             units='code_velocity', sampling_type='cell', force_override=True)
yt.add_field(('gas', 'vel_cyl_y'), function=_vel_cyl_y,
             units='code_velocity', sampling_type='cell', force_override=True)

In [ ]:
def _enthalpy(field, data):
    rho = data[('athena_pp', 'rho')]
    P = data[('athena_pp', 'press')]
    c = data.ds.quan(1.0, 'code_velocity')
    return 1 + 4*P/(rho*c**2)


yt.add_field(('gas', 'enthalpy'), function=_enthalpy, units="", sampling_type="cell", dimensions=yt.units.dimensionless)

def _entropy(field, data):
    rho = data[('athena_pp', 'rho')]
    P = data[('athena_pp', 'press')]
    c = data.ds.quan(1.0, 'code_velocity')
    return P/rho**(4/3)

yt.add_field(('gas', 'entropy'), function=_entropy, units="code_length**3/(code_mass**(1/3)*code_time**2)", sampling_type="cell")

In [ ]:

import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# 1D velocity profile at a fixed azimuthal angle
theta_vals = np.linspace(0, np.pi/2, 10)  # example theta values
z_val     = 0.5   #doesnt matter

ds_entr = yt.load('/scratch/aripoll/athena_out/outputs/jet_blast.out1.01800.athdf')


r_shock_entropy = []  # list of lists: 1 or more radii per theta

for theta_val in theta_vals:
    ray = ds_entr.ortho_ray(0, (theta_val, z_val))

    df_entr = ray.to_dataframe([('index', 'r'), ('gas', 'entropy')])
    df_entr = df_entr.sort_values('r').reset_index(drop=True)

    if len(df_entr) < 2:
        r_shock_entropy.append([np.nan])
        continue

    # Shock has hit the surface: outer boundary entropy is still unshocked
    if np.log10(df_entr['entropy'].iloc[-1]) < 1:
        r_shock_entropy.append([1.5])
        print(f"Shock at surface for theta={theta_val:.3f} rad")
        continue

    dlogS_dr = np.gradient(np.log10(df_entr['entropy']), df_entr['r'])
    abs_grad = np.abs(dlogS_dr)

    peaks, _ = find_peaks(abs_grad, height= 100, distance = 10)
    radii = [float(df_entr['r'].iloc[i]) for i in peaks if df_entr['r'].iloc[i] > 0.2]

    if radii:
        print(f"Shock fronts at theta={theta_val:.3f} rad: {', '.join(f'{r:.3f}' for r in radii)} R_*")
        r_shock_entropy.append(radii)
    else:
        r_shock_entropy.append([np.nan])
    fig, ax = plt.subplots()
    ax.plot(df_entr['r'], df_entr['entropy'], label='Entropy')
    for r in radii:
        ax.axvline(r, color='r', linestyle='--', label=f'shock r={r:.3f}')
    ax.set_xlabel(r"$r \ [R_*]$")
    ax.set_ylabel(r"Entropy $\mathcal{S}$")
    ax.set_title(rf"Entropy at $\theta = {theta_val*180/np.pi:.3f}$°, $t = {float(ds.current_time):.2f}\ R_*/c$") 
    ax.legend()
    plt.show()

# Flatten for polar plot: duplicate theta for entries with two fronts
plot_theta = [th for th, rs in zip(theta_vals, r_shock_entropy) for _ in rs]
plot_r     = [r  for rs in r_shock_entropy for r in rs]

plt.figure(figsize=(20, 20))
plt.polar(plot_theta, plot_r, marker='o', color='g', linestyle='none')
plt.title(rf"Shock Radius from Entropy Gradient at $t = {float(ds.current_time):.2f}\ R_*/c$")
plt.legend()
plt.show()

## Video of the whole thing

In [ ]:
from pathlib import Path
ts = yt.load('/scratch/aripoll/athena_out/outputs/jet_blast.out1.*.athdf')
nx, ny = ts[0].domain_dimensions[0], ts[0].domain_dimensions[1]
frames_dir = Path('/scratch/aripoll/athena_out/frames')
frames_dir.mkdir(parents=True, exist_ok=True)

for i, ds in enumerate(ts[800:1100], start=800):
    r_shock_entropy = []  # list of lists: 1 or more radii per theta

    for theta_val in theta_vals:
        ray = ds.ortho_ray(0, (theta_val, z_val))

        df_entr = ray.to_dataframe([('index', 'r'), ('gas', 'entropy')])
        df_entr = df_entr.sort_values('r').reset_index(drop=True)

        if len(df_entr) < 2:
            r_shock_entropy.append([np.nan])
            continue

        # Shock has hit the surface: outer boundary entropy is still unshocked
        if np.log10(df_entr['entropy'].iloc[-1]) < 5:
            r_shock_entropy.append([1.5])
            # print(f"Shock at surface for theta={theta_val:.3f} rad")
            continue

        dlogS_dr = np.gradient(np.log10(df_entr['entropy']), df_entr['r'])
        abs_grad = np.abs(dlogS_dr)

        peaks, _ = find_peaks(abs_grad, height= 100, distance = 10)
        radii = [float(df_entr['r'].iloc[i]) for i in peaks if df_entr['r'].iloc[i] > 0.2]

        if radii:
            # print(f"Shock fronts at theta={theta_val:.3f} rad: {', '.join(f'{r:.3f}' for r in radii)} R_*")
            r_shock_entropy.append(radii)
        else:
            r_shock_entropy.append([np.nan])
        # fig, ax = plt.subplots()
        # ax.plot(df_entr['r'], abs_grad, label='Entropy')
        # for r in radii:
        #     ax.axvline(r, color='r', linestyle='--', label=f'shock r={r:.3f}')
        # ax.set_xlabel(r"$r \ [R_*]$")
        # ax.set_ylabel(r"Entropy $\mathcal{S}$")
        # ax.set_title(rf"Entropy at $\theta = {theta_val*180/np.pi:.3f}$°, $t = {float(ds.current_time):.2f}\ R_*/c$") 
        # ax.legend()
        # plt.show()

    # Flatten for polar plot: duplicate theta for entries with two fronts
    plot_theta = [th for th, rs in zip(theta_vals, r_shock_entropy) for _ in rs]
    plot_r     = [r  for rs in r_shock_entropy for r in rs]

    plt.figure(figsize=(20, 20))
    plt.polar(plot_theta, plot_r, marker='o', color='g', linestyle='none')
    plt.title(rf"Shock Radius from Entropy Gradient at $t = {float(ds.current_time):.2f}\ R_*/c$")
    plt.savefig(frames_dir / f"frame_{i:04d}.png", dpi=100, bbox_inches='tight')
    plt.close()

In [ ]:
#Now make a video using ffmpeg starting from the first frame frame_1400.png to frame_1499.png
frames_dir = Path('/scratch/aripoll/athena_out/frames')
frames_dir.mkdir(parents=True, exist_ok=True)
import subprocess
subprocess.run(['ffmpeg', '-y', '-framerate', '10', '-start_number', '800', '-i', str(frames_dir / 'frame_%04d.png'), '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-vf', 'pad=ceil(iw/2)*2:ceil(ih/2)*2', 'shock_evolution.mp4'], check=True)

from IPython.display import Video, display
display(Video('shock_evolution.mp4', embed=True))

### Velocity of points

In [ ]:
from pathlib import Path
ts = yt.load('/scratch/aripoll/athena_out/outputs/jet_blast.out1.*.athdf')
nx, ny = ts[0].domain_dimensions[0], ts[0].domain_dimensions[1]
frames_dir = Path('/scratch/aripoll/athena_out/frames')
frames_dir.mkdir(parents=True, exist_ok=True)

# Store the list of radii so we can get a velocity
shock_records = []  # list of tuples: (time, list of lists of shock radii per theta)
for i, ds in enumerate(ts[:1100], start=0):
    r_shock = []  # list of lists: 1 or more radii per theta
    for theta_val in theta_vals:
        ray = ds.ortho_ray(0, (theta_val, z_val))

        df_entr = ray.to_dataframe([('index', 'r'), ('gas', 'entropy')])
        df_entr = df_entr.sort_values('r').reset_index(drop=True)

        if len(df_entr) < 2:
            r_shock.append([np.nan])
            continue

        # Shock has hit the surface: outer boundary entropy is still unshocked
        if np.log10(df_entr['entropy'].iloc[-1]) < 5:
            r_shock.append([1.5])
            print(f"Shock at surface for theta={theta_val:.3f} rad")
            continue

        dlogS_dr = np.gradient(np.log10(df_entr['entropy']), df_entr['r'])
        abs_grad = np.abs(dlogS_dr)

        peaks, _ = find_peaks(abs_grad, height=100, distance=10)
        radii = [float(df_entr['r'].iloc[p]) for p in peaks if df_entr['r'].iloc[p] > 0.2]

        if radii:
            print(f"Shock fronts at theta={theta_val:.3f} rad: {', '.join(f'{r:.3f}' for r in radii)} R_*")
            r_shock.append(radii)
        else:
            r_shock.append([np.nan])
    shock_records.append((float(ds.current_time), r_shock))

print(f"Collected {len(shock_records)} timesteps")

velocities = []  # list of (theta, r_mid, v_shock)
for i in range(1, len(shock_records)):
    t0, shock0 = shock_records[i - 1]
    t1, shock1 = shock_records[i]
    dt = t1 - t0

    for theta_val, rs0, rs1 in zip(theta_vals, shock0, shock1):
        if np.isnan(rs0).all() or np.isnan(rs1).all():
            continue  # skip if either timestep has no shock front

        valid0 = sorted(r for r in rs0 if not np.isnan(r))
        valid1 = sorted(r for r in rs1 if not np.isnan(r))
        used = set()
        for r0 in valid0:
            dists = [(abs(r0 - r1), j, r1) for j, r1 in enumerate(valid1) if j not in used]
            if not dists:
                continue
            _, j, r1 = min(dists)
            used.add(j)
            velocities.append((theta_val, (r0 + r1) / 2, (r1 - r0) / dt))

print(f"Computed {len(velocities)} velocity measurements")

In [ ]:
thetas = np.array([v[0] for v in velocities])
radii  = np.array([v[1] for v in velocities])
vels   = np.array([v[2] for v in velocities])

vmax = np.nanpercentile(np.abs(vels), 95)

# Polar scatter: shock speed at each (theta, r) position
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={'projection': 'polar'})
sc = ax.scatter(thetas, radii, c=vels, cmap='RdBu_r', s=5, vmin=-vmax, vmax=vmax)
plt.colorbar(sc, ax=ax, label=r'$v_{\rm shock}\ [c]$', fraction=0.04)
ax.set_title(r"Shock Velocity $dr/dt$ [c]" + f"\n({len(shock_records)} timesteps)")
plt.tight_layout()
plt.show()

# Velocity vs. radius at theta = 0
theta0 = theta_vals[0]
mask = thetas == theta0
r0_arr = radii[mask]
v0_arr = vels[mask]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(r0_arr, v0_arr, s=10, c='steelblue', alpha=0.7)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_xlabel(r"$r\ [R_*]$")
ax.set_ylabel(r"$v_{\rm shock}\ [c]$")
ax.set_title(r"Shock Velocity vs. Radius at $\theta = 0$")
plt.tight_layout()
plt.show()